# Loading data functions and testing

I coded the savings of data without the use of functions so I transfer and test into functions now

In [5]:
import pandas as pd
from vportfolio.params import RAW, PROCESSED, DATA_KEYS, RAW_FILE

In [6]:
# load data used accross all models
def _load_shared(folder = PROCESSED):
    rebal_dates = pd.read_parquet(folder / "rebal_dates.parquet")["date"]
    cost_bp = pd.read_parquet(folder / "cost_bp.parquet")["cost_bp"]
    rf_daily = pd.read_parquet(folder / "rf_daily.parquet")["rf"]
    shared_data= {"rebal_dates" : rebal_dates, "cost_bp" : cost_bp, "rf_daily" : rf_daily}
    return shared_data

# this function has no logic in itself so there is no testing needed

In [7]:
# load the universe of ETFs
def load_universe(folder = PROCESSED):
    close = pd.read_parquet(folder / "close.parquet")
    open_ = pd.read_parquet(folder / "open.parquet")
    returns = pd.read_parquet(folder / "returns.parquet")
    eligible = pd.read_parquet(folder / "eligible.parquet")
    universe_data = {"close" : close, "open" : open_, "returns" : returns, "eligible": eligible}
    shared_data = _load_shared(folder)
    universe_data = {**universe_data, **shared_data}
    return universe_data

In [19]:
# function testing

# run the function
universe_data = load_universe(folder = PROCESSED)

# test the names
assert set(universe_data.keys()) == set(DATA_KEYS)

# the dataframes in the dictionary have the correct column names and dates as index
for key in ["open", "returns", "eligible"]:
    assert universe_data[key].columns.equals(universe_data["close"].columns), key
    assert universe_data[key].index.equals(universe_data["close"].index), key

# the cost data covers every ticker
assert set(universe_data["close"].columns) <= set(universe_data["cost_bp"].index) # lesser or equal to make sure nothing is missing

# rebalance dates exist in real trading days
assert universe_data["rebal_dates"].isin(universe_data["close"].index).all()

print("all tests passed")

all tests passed


In [30]:
# load data for benchmark tickers only
def load_benchmark(tickers = ["SPY", "AGG"], raw_file = RAW_FILE, folder = PROCESSED):
    raw_data = pd.read_parquet(raw_file)
    close = raw_data["Close"][tickers]
    open_ = raw_data["Open"][tickers]
    returns = close.pct_change(fill_method=None)
    eligible = close.notna()
    benchmark_data = {"close" : close, "open" : open_, "returns" : returns, "eligible": eligible}
    shared_data = _load_shared(folder)
    benchmark_data = {**benchmark_data, **shared_data}
    return benchmark_data

In [35]:
# function testing

# run the function
benchmark_data = load_benchmark(folder = PROCESSED)

# add a test to see if the tickers correspond to the columns
assert list(benchmark_data["close"].columns) == ["SPY", "AGG"]

# test the names
assert set(benchmark_data.keys()) == set(DATA_KEYS)

# the dataframes in the dictionary have the correct column names and dates as index
for key in ["open", "returns", "eligible"]:
    assert benchmark_data[key].columns.equals(benchmark_data["close"].columns), key
    assert benchmark_data[key].index.equals(benchmark_data["close"].index), key

# the cost data covers every ticker
assert set(benchmark_data["close"].columns) <= set(benchmark_data["cost_bp"].index) # lesser or equal to make sure nothing is missing

# rebalance dates exist in real trading days. more important here since we read from RAW file and not PROCESSED
assert benchmark_data["rebal_dates"].isin(benchmark_data["close"].index).all()

print("all tests passed")

all tests passed


no need to pipeline these functions. benchmark data are only used for the 6040 allocator